In [ ]:

!pip install folium geopy


   ---------- ----------------------------- 1/4 [geopy]
   ---------- ----------------------------- 1/4 [geopy]
   ------------------------------ --------- 3/4 [folium]
   ------------------------------ --------- 3/4 [folium]
   ---------------------------------------- 4/4 [folium]



In [1]:
import math
import urllib.parse
import urllib.request
import json
import webbrowser
import os

In [ ]:
# 1. Chuyển địa chỉ sang tọa độ
def dia_chi_sang_toa_do(dia_chi):
    print(f" Đang tìm tọa độ trên bản đồ cho: {dia_chi}...")
    # Thêm "Ho Chi Minh, Vietnam" để giới hạn tìm kiếm chuẩn xác hơn trong khu vực
    query = f"{dia_chi}, Ho Chi Minh, Vietnam"
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(query)}&format=json&limit=1"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data:
                return (float(data[0]['lat']), float(data[0]['lon']))
    except Exception as e:
        print("Lỗi kết nối bản đồ:", e)
    return None

In [7]:
# 2. TÌM ĐƯỜNG ĐI NGẮN NHẤT THỰC TẾ
def tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach):
    print(" Đang tính toán lộ trình đường đi ngắn nhất...")
    url = f"http://router.project-osrm.org/route/v1/driving/{toa_do_quan[1]},{toa_do_quan[0]};{toa_do_khach[1]},{toa_do_khach[0]}?overview=full&geometries=geojson"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data['code'] == 'Ok':
                km_thuc_te = round(data['routes'][0]['distance'] / 1000, 2)
                phut_thuc_te = round(data['routes'][0]['duration'] / 60, 1)
                
                route_coords = data['routes'][0]['geometry']['coordinates']
                toa_do_ve_duong = [[lat, lon] for lon, lat in route_coords]
                
                return toa_do_ve_duong, km_thuc_te, phut_thuc_te
    except Exception as e:
        print("Lỗi tìm đường:", e)
    return None, None, None

def trang_thai_giao_hang(km):
    return "Đang giao" if km <= 10 else "Đã hủy"

In [11]:
def tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km, phut, mang_toa_do_duong_di):
    duong_di_json = json.dumps(mang_toa_do_duong_di)

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Bản đồ Lộ Trình Giao Hàng</title>
        <meta charset="utf-8" />
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

        <style>
            body {{ font-family: Arial; text-align:center; background:#f4f4f4; }}

            #map {{
                height:70vh;
                width:90%;
                margin:auto;
                border:2px solid #ccc;
                border-radius:8px;
            }}

            .info-box {{
                background:white;
                width:90%;
                margin:15px auto;
                padding:15px;
                border-radius:8px;
                box-shadow:0 2px 5px rgba(0,0,0,0.2);
            }}

            .bottom-box {{
                width:90%;
                margin:20px auto;
                display:flex;
                justify-content:space-between;
            }}

            .card {{
                width:45%;
                background:white;
                padding:15px;
                border-radius:8px;
                box-shadow:0 2px 5px rgba(0,0,0,0.2);
            }}

            h2 {{color:#e74c3c}}
        </style>

    </head>

    <body>

        <div class="info-box">
            <h2>🛵 Lộ trình giao hàng FastFood</h2>
            <p>Điểm đến: <b>{dia_chi_khach}</b></p>
            <p>Khoảng cách thực tế: <b>{km} km</b></p>
        </div>

        <div id="map"></div>

        <div class="bottom-box">

            <div class="card">
                <h3>📏 Khoảng cách</h3>
                <h2>{km} km</h2>
            </div>

            <div class="card">
                <h3>⏱ Thời gian còn lại</h3>
                <h2 id="countdown"></h2>
            </div>

        </div>

        <script>

            var map = L.map('map').setView([{toa_do_quan[0]}, {toa_do_quan[1]}], 13);

            L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
                attribution:'© OpenStreetMap'
            }}).addTo(map);

            var shopIcon = L.icon({{
                iconUrl:'https://cdn-icons-png.flaticon.com/512/3448/3448650.png',
                iconSize:[40,40]
            }});

            var homeIcon = L.icon({{
                iconUrl:'https://cdn-icons-png.flaticon.com/512/1946/1946436.png',
                iconSize:[40,40]
            }});

            L.marker([{toa_do_quan[0]}, {toa_do_quan[1]}],{{icon:shopIcon}})
            .addTo(map)
            .bindPopup("🍔 Cửa hàng FastFood Universe")
            .openPopup();

            L.marker([{toa_do_khach[0]}, {toa_do_khach[1]}],{{icon:homeIcon}})
            .addTo(map)
            .bindPopup("🏠 Khách hàng<br>{dia_chi_khach}");

            var routeCoords = {duong_di_json};

            var polyline = L.polyline(routeCoords,{{
                color:'#3498db',
                weight:6,
                opacity:0.8
            }}).addTo(map);

            map.fitBounds(polyline.getBounds(),{{padding:[30,30]}});

            // =============================
            // ĐẾM NGƯỢC 10 PHÚT
            // =============================

            var timeLeft = 10 * 60;

            function updateCountdown(){{
                var minutes = Math.floor(timeLeft/60);
                var seconds = timeLeft%60;

                document.getElementById("countdown").innerHTML =
                    minutes + "p " + (seconds<10?"0":"") + seconds + "s";

                if(timeLeft > 0){{
                    timeLeft--;
                }}
            }}

            setInterval(updateCountdown,1000);
            updateCountdown();

        </script>

    </body>
    </html>
    """

    file_name = "ban_do_lo_trinh_thuc_te.html"

    with open(file_name,"w",encoding="utf-8") as f:
        f.write(html_content)

    webbrowser.open('file://' + os.path.realpath(file_name))

In [12]:
# CHƯƠNG TRÌNH CHÍNH
toa_do_quan = (10.7634, 106.6821)  # FastFood Universe (227 Nguyễn Văn Cừ, Quận 5)
print(" CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE")
print(" Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.")
print("-" * 50)

# 1. NHẬP ĐỊA CHỈ TỰ DO
dia_chi_khach = input(" Nhập địa chỉ nhận hàng của bạn (VD: Chợ Bến Thành, Landmark 81, 123 Lê Lợi...): ").strip()
toa_do_khach = dia_chi_sang_toa_do(dia_chi_khach)

if not toa_do_khach:
    print(" Lỗi: Không tìm được địa chỉ trên bản đồ. Vui lòng thử nhập chi tiết hơn (gồm số nhà, tên đường, phường, quận).")
else:
    duong_di, km_thuc, phut_thuc = tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach)

    if not duong_di:
        print(" Lỗi: Không tìm được lộ trình giao thông đến địa chỉ này.")
    else:
        # 2. KIỂM TRA ĐIỀU KIỆN 10KM (Sử dụng khoảng cách thực tế)
        if km_thuc > 10:
            print(f"\n TỪ CHỐI ĐƠN HÀNG:")
            print(f"Khoảng cách đến chỗ bạn là {km_thuc} km. Rất tiếc, cửa hàng chỉ giao trong phạm vi 10 km đổ lại để đảm bảo chất lượng món ăn!")
        else:
            # 3. NẾU <= 10KM THÌ TIẾN HÀNH ĐẶT ĐƠN VÀ VẼ BẢN ĐỒ
            ket_qua = {
                "ten_quan": "FastFood Universe",
                "dia_chi_khach": dia_chi_khach,
                "khoang_cach_thuc_te_km": km_thuc,
                "thoi_gian_lai_xe_phut": phut_thuc,
                "trang_thai": trang_thai_giao_hang(km_thuc)
            }
            
            print("\n THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):")
            print(json.dumps(ket_qua, ensure_ascii=False, indent=4))
            tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km_thuc, phut_thuc, duong_di)

 CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE
 Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.
--------------------------------------------------
 Đang tìm tọa độ trên bản đồ cho: 23 le van sy...
 Đang tính toán lộ trình đường đi ngắn nhất...

 THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):
{
    "ten_quan": "FastFood Universe",
    "dia_chi_khach": "23 le van sy",
    "khoang_cach_thuc_te_km": 7.33,
    "thoi_gian_lai_xe_phut": 7.7,
    "trang_thai": "Đang giao"
}
